In [1]:
import torch
import numpy as np
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, f1_score, confusion_matrix,balanced_accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
import mlflow
import lightgbm as lgb

d:\anaconda\envs\genai-env\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
mlflow.set_experiment("binary_classification_experiment")

<Experiment: artifact_location='file:d:/fourth_year/graduation_project/JGuard/defenders/mult-iturn/NBF/mlruns/1', creation_time=1776014223598, experiment_id='1', last_update_time=1776014223598, lifecycle_stage='active', name='binary_classification_experiment', tags={}, trace_location=None, workspace='default'>

In [16]:
def log_trained_model(
    model,y_pred,y_test,features,params: dict,run_name: str = "run",model_name: str = "model",plot_name: str = "confusion_matrix.png"):
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        mlflow.log_param("features", str(features))

        # Metrics
        bal_acc = balanced_accuracy_score(y_test, y_pred)
        f1_score_val = f1_score(y_test, y_pred, average='macro')
        classification_report_val = classification_report(y_test, y_pred, output_dict=True)
        mlflow.log_metric("balanced_accuracy", bal_acc)
        mlflow.log_metric("f1_score", f1_score_val)
        mlflow.log_dict(classification_report_val, "classification_report.json")
        cm = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(6, 4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
        plt.title("Confusion Matrix")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.savefig(plot_name)
        plt.close()
        mlflow.log_artifact(plot_name)

        mlflow.sklearn.log_model(model, model_name)

        print(f"Run complete. Balanced Accuracy: {bal_acc:.4f}")

        return {
            "balanced_accuracy": bal_acc,
            "f1_score": f1_score_val,
            "confusion_matrix": cm.tolist()  
        }

In [2]:
data=torch.load("./train_data_with_context/all_conversations.pt")
data2=torch.load("./train_data_with_context/all_conversations2.pt")

In [3]:
data[0][0].keys()

dict_keys(['x_t', 'zt', 'score', 'y'])

In [4]:
labels_map={
    1:0,
    2:0,
    3:0,
    4:0,
    5:1
}

In [5]:
X = []
y=[]
for convo in data:
    for turn in convo:
        X.append(torch.concat([turn["x_t"], turn["zt"]], dim=0))
        y.append(labels_map[turn["score"]])

In [6]:
X2 = []
y2=[]
for convo in data2:
    for turn in convo:
        X2.append(torch.concat([turn["x_t"], turn["ut"]], dim=0))
        y2.append(labels_map[turn["score"]])

In [7]:
from collections import Counter
c=Counter(y)
c

Counter({0: 26362, 1: 2617})

In [8]:
# high imbalance ratio
ratio=c[0]/c[1]
ratio

10.07336645013374

In [9]:
X = np.array(X)
y = np.array(y)
X2 = np.array(X2)
y2 = np.array(y2)

In [10]:
X_train1, X_test1, y_train1, y_test1 = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2
)

In [11]:
pca = PCA(n_components=0.95)  # keep 95% of variance
pca.fit_transform(X_train2)
# save the PCA model
import joblib
joblib.dump(pca, "pca_model.pkl")

['pca_model.pkl']

In [13]:
weights = {0: 1.0, 1: 8.0}  
svm_balanced = SVC(kernel='rbf', C=1.0, gamma='scale', class_weight=weights)

y_train_pred = svm_balanced.fit(X_train1, y_train1).predict(X_train1)
y_test_pred = svm_balanced.predict(X_test1)

In [14]:
print("train_f1_score:", f1_score(y_train1, y_train_pred, average='macro'))
print("Balanced f1_score:", f1_score(y_test1, y_test_pred, average='macro'))
print(classification_report(y_test1, y_test_pred))

train_f1_score: 0.8348007990314081
Balanced f1_score: 0.7182060958637897
              precision    recall  f1-score   support

           0       0.97      0.90      0.93      5273
           1       0.40      0.68      0.50       523

    accuracy                           0.88      5796
   macro avg       0.68      0.79      0.72      5796
weighted avg       0.92      0.88      0.89      5796



In [17]:
log_trained_model(svm_balanced,y_test_pred,y_test1,features="Xt_Zt",params={"model":"SVM","kernel":"rbf","C":1.0,"gamma":"scale","class_weight":"weights = '{0: 1.0, 1: 8.0}'"},run_name="SVM_rbf_balanced",model_name="svm_rbf_different",plot_name="svm_rbf_balanced_cm.png")

2026/04/12 21:02:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/12 21:02:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/12 21:02:33 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\KARIMM~1\AppData\Local\Temp\tmpdtic8vj4\model\model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.7.1', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 


Run complete. Balanced Accuracy: 0.7900


{'balanced_accuracy': 0.789993324338172,
 'f1_score': 0.7182060958637897,
 'confusion_matrix': [[4742, 531], [167, 356]]}

In [ ]:
# from sklearn.model_selection import GridSearchCV

# params = {
#     'C': [0.1, 1, 10],
#     'kernel': ['linear', 'rbf'],
#     'gamma': ['scale', 'auto']
# }

# grid = GridSearchCV(SVC(), params, cv=5)
# grid.fit(X_train1, y_train1)

# print("Best params:", grid.best_params_)

Best params: {'C': 1, 'gamma': 'scale', 'kernel': 'rbf'}


# with dimensionality reduction

In [23]:
pca = PCA(n_components=0.95)  # keep 95% of variance
X_train_pca = pca.fit_transform(X_train1)
print("Number of components:", pca.n_components_)
print("Total explained variance:", pca.explained_variance_ratio_.sum())

weights = {0: 1.0, 1: 7.0}
model = SVC(kernel='rbf', C=1.0, gamma='scale', class_weight=weights)  

y_train_pred=model.fit(X_train_pca, y_train1).predict(X_train_pca)
y_test_pred = model.predict(pca.transform(X_test1))

Number of components: 213
Total explained variance: 0.9503398


In [21]:
print(classification_report(y_test1, y_test_pred))

              precision    recall  f1-score   support

           0       0.97      0.87      0.92      5273
           1       0.35      0.73      0.48       523

    accuracy                           0.85      5796
   macro avg       0.66      0.80      0.70      5796
weighted avg       0.91      0.85      0.88      5796



In [24]:
log_trained_model(run_name="SVM_pca_different_weights", model=model, y_pred=y_test_pred, y_test=y_test1, features="PCAXt_Zt", 
                params={"model":"SVM","kernel":"rbf","C":1.0,"gamma":"scale","class_weight":"{0: 1.0, 1: 7.0}"},
                model_name="svm_rbf_different", plot_name="svm_rbf_different_cm.png")

2026/04/12 21:08:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/12 21:08:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/12 21:08:53 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\KARIMM~1\AppData\Local\Temp\tmptp01rxfl\model\model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.7.1', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 


Run complete. Balanced Accuracy: 0.7850


{'balanced_accuracy': 0.7849923797374627,
 'f1_score': 0.7218843294140127,
 'confusion_matrix': [[4780, 493], [176, 347]]}

# trial 2

In [25]:
model = SVC(kernel='rbf', C=1.0, gamma='scale', class_weight="balanced") 

# Train
y_train_pred =model.fit(X_train2, y_train2).predict(X_train2)

# Predict
y_test_pred = model.predict(X_test2)

# Evaluate
print("Accuracy:", accuracy_score(y_test2, y_test_pred))
print(classification_report(y_test2, y_test_pred))

Accuracy: 0.879399585921325
              precision    recall  f1-score   support

           0       0.97      0.90      0.93      5273
           1       0.40      0.68      0.50       523

    accuracy                           0.88      5796
   macro avg       0.68      0.79      0.72      5796
weighted avg       0.91      0.88      0.89      5796



In [29]:
log_trained_model(run_name="SVM_balanced_weights", model=model, y_pred=y_test_pred, y_test=y_test2, features="Xt_ut", 
                params={"model":"SVM","kernel":"rbf","C":1.0,"gamma":"scale","class_weight":"balanced"},
                model_name="svm_rbf_balanced", plot_name="svm_rbf_balanced_cm.png")

2026/04/12 21:23:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/12 21:23:45 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/12 21:23:49 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\KARIMM~1\AppData\Local\Temp\tmpyojckg66\model\model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.7.1', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 


Run complete. Balanced Accuracy: 0.7899


{'balanced_accuracy': 0.7898985016565867,
 'f1_score': 0.7179746795614543,
 'confusion_matrix': [[4741, 532], [167, 356]]}

In [31]:
pca = PCA(n_components=0.95)  # keep 95% of variance
X_train_pca = pca.fit_transform(X_train2)
print("Number of components:", pca.n_components_)
print("Total explained variance:", pca.explained_variance_ratio_.sum())

# Create SVM model
model = SVC(kernel='rbf', C=1.0, gamma='scale', class_weight="balanced")  

# Train
model.fit(X_train_pca, y_train2)

# Predict
y_test_pred = model.predict(pca.transform(X_test2))

# Evaluate
print("Accuracy:", accuracy_score(y_test2, y_test_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_test2, y_test_pred))
print("F1 Score:", f1_score(y_test2, y_test_pred, average='macro'))
print(classification_report(y_test2, y_test_pred))

Number of components: 352
Total explained variance: 0.9500107
Accuracy: 0.8814699792960663
Balanced Accuracy: 0.7832855714689249
F1 Score: 0.7176270057329805
              precision    recall  f1-score   support

           0       0.96      0.90      0.93      5273
           1       0.40      0.66      0.50       523

    accuracy                           0.88      5796
   macro avg       0.68      0.78      0.72      5796
weighted avg       0.91      0.88      0.89      5796



In [32]:
log_trained_model(run_name="SVM_balanced_weights_pca", model=model, y_pred=y_test_pred, y_test=y_test2, features="pca(Xt_ut)", 
                params={"model":"SVM","kernel":"rbf","C":1.0,"gamma":"scale","class_weight":"balanced"},
                model_name="svm_rbf_balanced", plot_name="cm.png")

2026/04/12 21:29:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/12 21:29:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/12 21:29:06 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\KARIMM~1\AppData\Local\Temp\tmpepmwm82s\model\model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.7.1', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 


Run complete. Balanced Accuracy: 0.7833


{'balanced_accuracy': 0.7832855714689249,
 'f1_score': 0.7176270057329805,
 'confusion_matrix': [[4762, 511], [176, 347]]}

# lightgbm

In [49]:
neg = (y_train1 == 0).sum()
pos = (y_train1 == 1).sum()

scale_pos_weight = neg / pos
print(scale_pos_weight)

10.071155682903534


In [ ]:
lgbm = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=64,
    max_depth=-1,
    scale_pos_weight=scale_pos_weight,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

lgbm.fit(X_train1, y_train1)

In [51]:
lgbm2 = lgb.LGBMClassifier(
    n_estimators=3000,          # more trees, lower learning rate
    learning_rate=0.01,
    num_leaves=128,             # more expressive (important for embeddings)
    max_depth=-1,

    subsample=0.8,
    colsample_bytree=0.7,

    min_child_samples=30,       # prevents overfitting
    reg_alpha=0.1,              # L1 regularization
    reg_lambda=0.1,             # L2 regularization

    scale_pos_weight=scale_pos_weight,

    random_state=42
)

In [52]:
lgbm2.fit(
    X_train1, y_train1,
    eval_set=[(X_test1, y_test2)],
    eval_metric="aucpr",
    callbacks=[lgb.early_stopping(100)]
)

[LightGBM] [Info] Number of positive: 2094, number of negative: 21089
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.429514 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 391680
[LightGBM] [Info] Number of data points in the train set: 23183, number of used features: 1536
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.090325 -> initscore=-2.309675
[LightGBM] [Info] Start training from score -2.309675
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[668]	valid_0's binary_logloss: 0.207636


,boosting_type,'gbdt'
,num_leaves,128
,max_depth,-1
,learning_rate,0.01
,n_estimators,3000
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,30


In [53]:
y_test_pred = lgbm.predict(X_test1)

# Evaluate
print("Accuracy:", accuracy_score(y_test1, y_test_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_test1, y_test_pred))
print("F1 Score:", f1_score(y_test1, y_test_pred, average='macro'))
print(classification_report(y_test1, y_test_pred))

Accuracy: 0.9194271911663217
Balanced Accuracy: 0.6499917143469437
F1 Score: 0.6875730285227781
              precision    recall  f1-score   support

           0       0.94      0.98      0.96      5273
           1       0.60      0.32      0.42       523

    accuracy                           0.92      5796
   macro avg       0.77      0.65      0.69      5796
weighted avg       0.91      0.92      0.91      5796



d:\anaconda\envs\genai-env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [54]:
log_trained_model(run_name="lgbm_2", model=lgbm2, y_pred=y_test_pred, y_test=y_test1, features="xt_Zt", 
                params={"model":"LGBM","learning_rate":0.01,"n_estimators":3000,"num_leaves":128,"max_depth":-1,"scale_pos_weight":ratio,"subsample":0.8,"colsample_bytree":0.7,"min_child_samples":30,"reg_alpha":0.1,"reg_lambda":0.1},
                model_name="lgbm", plot_name="cm.png")

2026/04/12 22:04:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/12 22:04:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/12 22:04:56 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\KARIMM~1\AppData\Local\Temp\tmp4n_soamh\model\model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.7.1', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 


Run complete. Balanced Accuracy: 0.6500


{'balanced_accuracy': 0.6499917143469437,
 'f1_score': 0.6875730285227781,
 'confusion_matrix': [[5161, 112], [355, 168]]}

# lgbm with classifier

In [67]:
pca = PCA(n_components=0.95)  # keep 95% of variance
X_train_pca = pca.fit_transform(X_train1)
print("Number of components:", pca.n_components_)
print("Total explained variance:", pca.explained_variance_ratio_.sum())

lgbm = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=64,
    max_depth=-1,
    class_weight={0:1,1:10},
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

lgbm.fit(X_train_pca, y_train1) 

y_train_pred=lgbm.predict(X_train_pca)
y_test_pred = lgbm.predict(pca.transform(X_test1))

Number of components: 213
Total explained variance: 0.9503398
[LightGBM] [Info] Number of positive: 2094, number of negative: 21089
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.017869 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 54315
[LightGBM] [Info] Number of data points in the train set: 23183, number of used features: 213
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.498227 -> initscore=-0.007090
[LightGBM] [Info] Start training from score -0.007090


d:\anaconda\envs\genai-env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\anaconda\envs\genai-env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [69]:
# Evaluate
print("Accuracy:", accuracy_score(y_test1, y_test_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_test1, y_test_pred))
print("F1 Score:", f1_score(y_test1, y_test_pred, average='macro'))
print(classification_report(y_test1, y_test_pred))

Accuracy: 0.9187370600414079
Balanced Accuracy: 0.6280824170464712
F1 Score: 0.6671684490364962
              precision    recall  f1-score   support

           0       0.93      0.98      0.96      5273
           1       0.61      0.27      0.38       523

    accuracy                           0.92      5796
   macro avg       0.77      0.63      0.67      5796
weighted avg       0.90      0.92      0.90      5796



In [63]:
log_trained_model(run_name="lgbm_pca", model=lgbm2, y_pred=y_test_pred, y_test=y_test1, features="pca(xt_Zt)", 
                params={"n_estimators": 1000, "learning_rate": 0.05, "num_leaves": 64, "max_depth": -1, "scale_pos_weight": 15,
                         "subsample": 0.8, "colsample_bytree": 0.8, "random_state": 42},
                model_name="lgbm", plot_name="cm.png")

2026/04/12 22:16:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/12 22:16:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/12 22:16:41 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\KARIMM~1\AppData\Local\Temp\tmpev3z52fh\model\model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.7.1', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 


Run complete. Balanced Accuracy: 0.6410


{'balanced_accuracy': 0.6409926248622533,
 'f1_score': 0.6809623391081558,
 'confusion_matrix': [[5177, 96], [366, 157]]}